# Run the experiment on DKT with the Junyi daatset

## Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

BASE_DIR = "/content/drive/MyDrive/education-ml-research"
JUNYI_DIR = os.path.join(BASE_DIR, "Junyi")
RESULTS_DIR = os.path.join(JUNYI_DIR, "reliability_experiment_irt")

os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print("Device:", device)
print("Junyi directory exists:", os.path.exists(JUNYI_DIR))

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
PyTorch version: 2.11.0+cu130
Device: cuda
Junyi directory exists: True
GPU: NVIDIA A100-SXM4-40GB


## Load datasets

In [ ]:
# File paths
JUNYI_DATA_PATH = os.path.join(JUNYI_DIR, "Log_Problem.csv")
IRT_PATH = os.path.join(JUNYI_DIR, "student_irt_ability_groups.csv")

# Verify paths
assert os.path.exists(JUNYI_DATA_PATH), f"Missing file: {JUNYI_DATA_PATH}"
assert os.path.exists(IRT_PATH), f"Missing file: {IRT_PATH}"

# Load datasets
junyi_df = pd.read_csv(JUNYI_DATA_PATH)
irt_df = pd.read_csv(IRT_PATH)

# Inspect interaction data
print("=== Junyi interaction dataset ===")
print("Shape:", junyi_df.shape)
print("Columns:", junyi_df.columns.tolist())
display(junyi_df.head())

print("\n=== IRT ability groups ===")
print("Shape:", irt_df.shape)
print("Columns:", irt_df.columns.tolist())
display(irt_df.head())

# Basic student coverage
print("\n=== Student coverage ===")
print("Unique interaction students:", junyi_df["user_id"].nunique()
      if "user_id" in junyi_df.columns else "Check student ID column")
print("Unique IRT students:", irt_df.iloc[:, 0].nunique())

=== Junyi interaction dataset ===
Shape: (16217311, 14)
Columns: ['timestamp_TW', 'uuid', 'ucid', 'upid', 'problem_number', 'exercise_problem_repeat_session', 'is_correct', 'total_sec_taken', 'total_attempt_cnt', 'used_hint_cnt', 'is_hint_used', 'is_downgrade', 'is_upgrade', 'level']


,timestamp_TW,uuid,ucid,upid,problem_number,exercise_problem_repeat_session,is_correct,total_sec_taken,total_attempt_cnt,used_hint_cnt,is_hint_used,is_downgrade,is_upgrade,level
0,2019-05-26 21:00:00 UTC,FLy+lviglNR5Y1l0Xiijnl6QHySBcpKHJLCtQ6ogm2Q=,KDOmuTrY/IJzDP4kIgIYCBiGyTymsJ8Iy4cDB35WGYg=,Vbs92l4JmdiWkUEm/iahxnUTaac2oN1IlUtXB7JcfoE=,18,2,True,33,1,0,False,False,True,3
1,2019-05-17 16:30:00 UTC,+Gqj2nalc6M9fusyVECTC0AN7UQdDQTXESIuElkDltU=,COZ39Wo+uIUO2s7c2VGEHjJf6Vx0xifxVAiaeHtaTdk=,Ek+pIeHNNoEo0tGEq91eBcBmGgy3+A5RWhpj95zTyHM=,4,1,True,8,1,0,False,NaN,NaN,0
2,2019-05-15 19:15:00 UTC,6D5QN8j8ng/VR74ES3A0zqAj0bIFFyaKjKEj8ZyXjQ8=,TwyqyV1uJYlDAX8wX/PtTCVZEBo/APIVfTzzleGkNCQ=,1MBa2f5Qog4JBoAuUfJf0fxeJctdEirAqKgfsg246eI=,9,1,True,17,1,0,False,NaN,NaN,0
3,2019-05-05 14:45:00 UTC,GgTZuCqZXObthtK6GAwqvlHrTMm5pKHWeezQxL/pcKc=,tBo6ECyT8IlKAM8UhQHWkqv92PRLcSiwuerfC7vNX+w=,kdMy2nG+QVMjPkuaMEWs0yV/sYZVoG1vm7zM0fCy+qk=,2,1,True,10,1,0,False,NaN,NaN,0
4,2019-05-14 16:45:00 UTC,JMNKWoU0CkMSzgQ8bCnmCYlD8jEzAVge3lHMYLXKM2g=,vVpSKAMQbTMvtdERR0ksOeRmmaFt0R210t4Z//0RpPA=,jjPR8fmkLSFoCQQYB4g6kI8mgdcK3sKtMirKUvfmZIk=,6,1,True,98,1,0,False,NaN,NaN,0



=== IRT ability groups ===
Shape: (72758, 4)
Columns: ['uuid', 'theta_all', 'theta_quartile', 'theta_sd_group']


,uuid,theta_all,theta_quartile,theta_sd_group
0,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,3.095891,Q4,> +1.5 SD
1,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,-2.329677,Q1,< -1.5 SD
2,++E4TrlDYvGtPBg1edhkLXLEEbnfiAgAamPQ33vpW8M=,-0.632970,Q2,-1.0 to -0.5 SD
3,++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=,-0.265644,Q2,-0.5 to 0 SD
4,++GobOSWqrsaxoRg1bMN+T6biIJcgBXwuOH/ddq3DiU=,0.577118,Q3,+0.5 to +1.0 SD



=== Student coverage ===
Unique interaction students: Check student ID column
Unique IRT students: 72758


## Inspect datasets

In [ ]:
print("=== Junyi interaction data ===")
print("Rows:", f"{len(junyi_df):,}")
print("Unique students:", f"{junyi_df['uuid'].nunique():,}")
print("Unique skills (ucid):", f"{junyi_df['ucid'].nunique():,}")
print("Unique problems (upid):", f"{junyi_df['upid'].nunique():,}")

print("\nMissing values:")
print(
    junyi_df[
        ["uuid", "ucid", "is_correct", "timestamp_TW"]
    ].isna().sum()
)

print("\nResponse distribution:")
print(junyi_df["is_correct"].value_counts(dropna=False))

print("\n=== IRT ability groups ===")
print("IRT rows:", f"{len(irt_df):,}")
print("Unique IRT students:", f"{irt_df['uuid'].nunique():,}")
print("Missing theta:", f"{irt_df['theta_all'].isna().sum():,}")

print("\nQuartile counts:")
print(irt_df["theta_quartile"].value_counts(dropna=False).sort_index())

print("\nSD-group counts:")
print(irt_df["theta_sd_group"].value_counts(dropna=False))

# Compare student coverage without merging the full interaction dataset
interaction_students = set(junyi_df["uuid"].dropna().unique())
irt_students = set(irt_df["uuid"].dropna().unique())

print("\n=== Student overlap ===")
print("Students in both datasets:",
      f"{len(interaction_students & irt_students):,}")
print("Interaction students without IRT:",
      f"{len(interaction_students - irt_students):,}")
print("IRT students without interactions:",
      f"{len(irt_students - interaction_students):,}")

=== Junyi interaction data ===
Rows: 16,217,311
Unique students: 72,758
Unique skills (ucid): 1,326
Unique problems (upid): 25,785

Missing values:
uuid            0
ucid            0
is_correct      0
timestamp_TW    0
dtype: int64

Response distribution:
is_correct
True     11412558
False     4804753
Name: count, dtype: int64

=== IRT ability groups ===
IRT rows: 72,758
Unique IRT students: 72,758
Missing theta: 0

Quartile counts:
theta_quartile
Q1    18190
Q2    18189
Q3    18189
Q4    18190
Name: count, dtype: int64

SD-group counts:
theta_sd_group
-0.5 to 0 SD       15274
0 to +0.5 SD       14323
-1.0 to -0.5 SD    11968
+0.5 to +1.0 SD     9692
-1.5 to -1.0 SD     7060
< -1.5 SD           5423
+1.0 to +1.5 SD     5169
> +1.5 SD           3849
Name: count, dtype: int64

=== Student overlap ===
Students in both datasets: 72,758
Interaction students without IRT: 0
IRT students without interactions: 0


## Clean data and create student level splits

In [ ]:
from sklearn.model_selection import train_test_split

# Keep only columns required for DKT and ability-group evaluation.
# Use a separate, reduced dataframe to limit memory usage.
model_df = junyi_df[
    ["uuid", "ucid", "timestamp_TW", "is_correct"]
].copy()

# Remove unusable rows, if any
model_df = model_df.dropna(
    subset=["uuid", "ucid", "timestamp_TW", "is_correct"]
)

# Normalize types
model_df["is_correct"] = model_df["is_correct"].astype("int8")
model_df["timestamp_TW"] = pd.to_datetime(
    model_df["timestamp_TW"], utc=True, errors="coerce"
)
model_df = model_df.dropna(subset=["timestamp_TW"])

# Join the IRT group labels
irt_labels = irt_df[
    ["uuid", "theta_all", "theta_quartile", "theta_sd_group"]
].drop_duplicates("uuid")

model_df = model_df.merge(
    irt_labels,
    on="uuid",
    how="inner",
    validate="many_to_one"
)

# Sort by student and chronological interaction order
model_df.sort_values(
    ["uuid", "timestamp_TW"],
    kind="mergesort",
    inplace=True
)
model_df.reset_index(drop=True, inplace=True)

# Student-level stratified split
student_info = irt_labels[
    irt_labels["uuid"].isin(model_df["uuid"].unique())
].copy()

trainval_ids, test_ids = train_test_split(
    student_info["uuid"],
    test_size=0.20,
    random_state=SEED,
    stratify=student_info["theta_quartile"]
)

trainval_info = student_info[
    student_info["uuid"].isin(trainval_ids)
]

train_ids, val_ids = train_test_split(
    trainval_info["uuid"],
    test_size=0.10,
    random_state=SEED,
    stratify=trainval_info["theta_quartile"]
)

train_ids = set(train_ids)
val_ids = set(val_ids)
test_ids = set(test_ids)

# Assign each interaction to its student's split
model_df["split"] = np.select(
    [
        model_df["uuid"].isin(train_ids),
        model_df["uuid"].isin(val_ids),
        model_df["uuid"].isin(test_ids)
    ],
    ["train", "validation", "test"],
    default="excluded"
)

train_df = model_df[model_df["split"] == "train"].copy()
val_df = model_df[model_df["split"] == "validation"].copy()
test_df = model_df[model_df["split"] == "test"].copy()

# Report results
print("=== Cleaned dataset ===")
print("Rows:", f"{len(model_df):,}")
print("Students:", f"{model_df['uuid'].nunique():,}")
print("Skills:", f"{model_df['ucid'].nunique():,}")
print("Response values:", sorted(model_df["is_correct"].unique()))

print("\n=== Split summary ===")
for name, frame in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        f"{name:10s}: {frame['uuid'].nunique():,} students, "
        f"{len(frame):,} interactions"
    )

print("\n=== Student overlap checks ===")
print("Train ∩ validation:", len(train_ids & val_ids))
print("Train ∩ test:", len(train_ids & test_ids))
print("Validation ∩ test:", len(val_ids & test_ids))

print("\n=== Students by IRT quartile ===")
print(pd.DataFrame({
    "train": model_df[model_df["split"] == "train"]
        .groupby("theta_quartile")["uuid"].nunique(),
    "validation": model_df[model_df["split"] == "validation"]
        .groupby("theta_quartile")["uuid"].nunique(),
    "test": model_df[model_df["split"] == "test"]
        .groupby("theta_quartile")["uuid"].nunique()
}).fillna(0).astype(int))

=== Cleaned dataset ===
Rows: 16,217,311
Students: 72,758
Skills: 1,326
Response values: [np.int8(0), np.int8(1)]

=== Split summary ===
Train     : 52,385 students, 11,675,589 interactions
Validation: 5,821 students, 1,273,106 interactions
Test      : 14,552 students, 3,268,616 interactions

=== Student overlap checks ===
Train ∩ validation: 0
Train ∩ test: 0
Validation ∩ test: 0

=== Students by IRT quartile ===
                train  validation  test
theta_quartile                         
Q1              13097        1455  3638
Q2              13096        1455  3638
Q3              13096        1455  3638
Q4              13096        1456  3638


## Build skill vocabulary and encode for training data

In [ ]:
# Build skill vocabulary from training data only
skill2idx = {
    skill: idx
    for idx, skill in enumerate(
        sorted(train_df["ucid"].unique())
    )
}

train_skills = set(skill2idx.keys())
val_skills = set(val_df["ucid"].unique())
test_skills = set(test_df["ucid"].unique())

unseen_val = val_skills - train_skills
unseen_test = test_skills - train_skills

print("=== Skill vocabulary ===")
print("Training skills:", len(train_skills))
print("Validation skills:", len(val_skills))
print("Test skills:", len(test_skills))
print("Unseen validation skills:", len(unseen_val))
print("Unseen test skills:", len(unseen_test))

# Save mapping for reproducibility
skill_map_path = os.path.join(RESULTS_DIR, "junyi_skill2idx.csv")

pd.DataFrame({
    "ucid": list(skill2idx.keys()),
    "skill_idx": list(skill2idx.values())
}).to_csv(skill_map_path, index=False)

print("\nSaved skill mapping:", skill_map_path)

# Encode skill IDs as compact integer indices
for frame in (train_df, val_df, test_df):
    frame["skill_idx"] = frame["ucid"].map(skill2idx).astype("int16")

print("\nEncoded sample:")
display(
    train_df[
        ["uuid", "timestamp_TW", "ucid", "skill_idx", "is_correct",
         "theta_quartile", "theta_sd_group"]
    ].head()
)

assert not unseen_val, f"Unseen validation skills: {len(unseen_val)}"
assert not unseen_test, f"Unseen test skills: {len(unseen_test)}"
assert train_df["skill_idx"].notna().all()
assert val_df["skill_idx"].notna().all()
assert test_df["skill_idx"].notna().all()

print("\nVocabulary checks passed.")

=== Skill vocabulary ===
Training skills: 1326
Validation skills: 1326
Test skills: 1326
Unseen validation skills: 0
Unseen test skills: 0

Saved skill mapping: /content/drive/MyDrive/education-ml-research/Junyi/reliability_experiment_irt/junyi_skill2idx.csv

Encoded sample:


,uuid,timestamp_TW,ucid,skill_idx,is_correct,theta_quartile,theta_sd_group
0,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,693,1,Q4,> +1.5 SD
1,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,693,1,Q4,> +1.5 SD
2,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,693,1,Q4,> +1.5 SD
3,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,693,1,Q4,> +1.5 SD
4,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,693,1,Q4,> +1.5 SD



Vocabulary checks passed.


## Check duplicate interactions

In [ ]:
# Check exact duplicates among the four essential interaction fields
key_cols = ["uuid", "timestamp_TW", "ucid", "is_correct"]

print("=== Duplicate interaction checks ===")

exact_duplicate_count = train_df.duplicated(subset=key_cols).sum()
print("Exact duplicate rows in training:", f"{exact_duplicate_count:,}")

# Check repeated student-timestamp-skill combinations, regardless of response
student_time_skill = ["uuid", "timestamp_TW", "ucid"]

repeated_key_mask = train_df.duplicated(
    subset=student_time_skill, keep=False
)

print(
    "Rows in repeated student-timestamp-skill combinations:",
    f"{repeated_key_mask.sum():,}"
)

print(
    "Unique repeated combinations:",
    f"{train_df.loc[repeated_key_mask, student_time_skill].drop_duplicates().shape[0]:,}"
)

print("\nExample repeated combinations:")
display(
    train_df.loc[
        repeated_key_mask,
        key_cols
    ].sort_values(student_time_skill).head(20)
)

# Inspect the original raw data for one repeated combination
if repeated_key_mask.any():
    example = train_df.loc[repeated_key_mask, student_time_skill].iloc[0]

    raw_example = junyi_df[
        (junyi_df["uuid"] == example["uuid"]) &
        (junyi_df["timestamp_TW"] == example["timestamp_TW"]) &
        (junyi_df["ucid"] == example["ucid"])
    ]

    print("\nCorresponding original rows:")
    display(raw_example.head(10))

=== Duplicate interaction checks ===
Exact duplicate rows in training: 7,815,332
Rows in repeated student-timestamp-skill combinations: 11,315,723
Unique repeated combinations: 2,346,186

Example repeated combinations:


,uuid,timestamp_TW,ucid,is_correct
0,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
1,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
2,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
3,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
4,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
5,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,2019-05-22 16:45:00+00:00,FR4KFXSJBwuu1LqaZRVJs+gv6d7T8mqibchrQhNvAKc=,0
6,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,2019-05-22 16:45:00+00:00,FR4KFXSJBwuu1LqaZRVJs+gv6d7T8mqibchrQhNvAKc=,0
7,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,2019-05-22 16:45:00+00:00,FR4KFXSJBwuu1LqaZRVJs+gv6d7T8mqibchrQhNvAKc=,0
8,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,2019-05-22 16:45:00+00:00,FR4KFXSJBwuu1LqaZRVJs+gv6d7T8mqibchrQhNvAKc=,0
9,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,2019-05-22 16:45:00+00:00,FR4KFXSJBwuu1LqaZRVJs+gv6d7T8mqibchrQhNvAKc=,0



Corresponding original rows:


,timestamp_TW,uuid,ucid,upid,problem_number,exercise_problem_repeat_session,is_correct,total_sec_taken,total_attempt_cnt,used_hint_cnt,is_hint_used,is_downgrade,is_upgrade,level


In [ ]:
# Inspect the original rows for one repeated student-timestamp-skill key.
# This helps determine whether repeated rows represent different problems,
# repeated attempts, or exact duplicate records.

example = train_df.loc[
    repeated_key_mask,
    ["uuid", "timestamp_TW", "ucid"]
].iloc[0]

example_rows = junyi_df[
    (junyi_df["uuid"] == example["uuid"]) &
    (junyi_df["timestamp_TW"] == example["timestamp_TW"]) &
    (junyi_df["ucid"] == example["ucid"])
].copy()

print("=== Original rows for the example combination ===")
print("Number of matching rows:", len(example_rows))

display(
    example_rows[
        [
            "timestamp_TW",
            "uuid",
            "ucid",
            "upid",
            "problem_number",
            "exercise_problem_repeat_session",
            "is_correct",
            "total_attempt_cnt",
            "total_sec_taken",
            "used_hint_cnt",
        ]
    ].head(30)
)

print("\n=== Distinct values within this combination ===")
for col in [
    "upid",
    "problem_number",
    "exercise_problem_repeat_session",
    "is_correct",
    "total_attempt_cnt",
    "total_sec_taken",
]:
    print(f"\n{col}:")
    print(example_rows[col].value_counts(dropna=False).head(10))

=== Original rows for the example combination ===
Number of matching rows: 0


,timestamp_TW,uuid,ucid,upid,problem_number,exercise_problem_repeat_session,is_correct,total_attempt_cnt,total_sec_taken,used_hint_cnt



=== Distinct values within this combination ===

upid:
Series([], Name: count, dtype: int64)

problem_number:
Series([], Name: count, dtype: int64)

exercise_problem_repeat_session:
Series([], Name: count, dtype: int64)

is_correct:
Series([], Name: count, dtype: int64)

total_attempt_cnt:
Series([], Name: count, dtype: int64)

total_sec_taken:
Series([], Name: count, dtype: int64)


In [ ]:
# Get the example key from the repeated rows
example = train_df.loc[
    repeated_key_mask,
    ["uuid", "timestamp_TW", "ucid"]
].iloc[0]

print("Example key:")
print("Student:", example["uuid"])
print("Timestamp:", repr(example["timestamp_TW"]))
print("Skill:", example["ucid"])

# Match using student and skill first, then inspect timestamps
candidate_rows = junyi_df[
    (junyi_df["uuid"] == example["uuid"]) &
    (junyi_df["ucid"] == example["ucid"])
].copy()

print("\nMatching rows by student and skill:", len(candidate_rows))

if len(candidate_rows) > 0:
    candidate_rows["timestamp_parsed"] = pd.to_datetime(
        candidate_rows["timestamp_TW"], utc=True, errors="coerce"
    )

    example_timestamp = pd.to_datetime(
        example["timestamp_TW"], utc=True, errors="coerce"
    )

    matching_rows = candidate_rows[
        candidate_rows["timestamp_parsed"] == example_timestamp
    ]

    print("Matching rows after timestamp normalization:", len(matching_rows))

    display(
        matching_rows[
            [
                "timestamp_TW", "uuid", "ucid", "upid",
                "problem_number", "exercise_problem_repeat_session",
                "is_correct", "total_attempt_cnt", "total_sec_taken"
            ]
        ].head(20)
    )
else:
    print("No original rows found for this student-skill pair.")

Example key:
Student: ++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=
Timestamp: Timestamp('2018-12-22 08:30:00+0000', tz='UTC')
Skill: VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=

Matching rows by student and skill: 5
Matching rows after timestamp normalization: 5


,timestamp_TW,uuid,ucid,upid,problem_number,exercise_problem_repeat_session,is_correct,total_attempt_cnt,total_sec_taken
5851364,2018-12-22 08:30:00 UTC,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,wgsncTN/ZG/jV1BVhOb0ExV10Y7nbx4HDXgpqfV98Ls=,2,1,True,1,17
10682047,2018-12-22 08:30:00 UTC,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,uIHmyJc0Ia07OXiS1k2wayrEA57AKPTypHaTYtKc4Po=,5,1,True,1,7
11828870,2018-12-22 08:30:00 UTC,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,SZt4MEfoLNcDjOc4qWBsd5Lw5M03DA0f22Baij04cbg=,1,1,True,1,193
13798435,2018-12-22 08:30:00 UTC,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,hrdzJpcOlqUZbvMB6A0P2fHVvn6US4bPm05s/PsjRi4=,3,1,True,1,55
15461504,2018-12-22 08:30:00 UTC,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,s2T1bZCtwOT2xgCGkHFRiqlSX8MHEOiW36tBJRXYj4E=,4,1,True,1,30


In [ ]:
# Check exact duplicate rows across every column
exact_duplicate_mask = junyi_df.duplicated(keep="first")

print("Total rows:", len(junyi_df))
print("Exact duplicate rows beyond the first occurrence:", exact_duplicate_mask.sum())
print("Percentage of dataset:", round(100 * exact_duplicate_mask.mean(), 4), "%")

if exact_duplicate_mask.any():
    print("\nExample exact duplicates:")
    duplicate_examples = junyi_df.loc[exact_duplicate_mask].head(10)
    display(duplicate_examples)
else:
    print("\nNo exact duplicate rows found across all columns.")

Total rows: 16217311
Exact duplicate rows beyond the first occurrence: 0
Percentage of dataset: 0.0 %

No exact duplicate rows found across all columns.


## Quantify timestamp ties

In [ ]:
# Count records sharing the same student and timestamp
timestamp_stats = (
    model_df.groupby(["uuid", "timestamp_TW"], sort=False)
    .size()
    .rename("n_records")
)

print("Student-timestamp groups:", len(timestamp_stats))
print("Groups with multiple records:", (timestamp_stats > 1).sum())
print(
    "Percentage of groups with ties:",
    round(100 * (timestamp_stats > 1).mean(), 2),
    "%"
)
print(
    "Records belonging to tied timestamp groups:",
    timestamp_stats[timestamp_stats > 1].sum()
)
print(
    "Percentage of all records in tied groups:",
    round(
        100 * timestamp_stats[timestamp_stats > 1].sum() / len(model_df),
        2
    ),
    "%"
)

print("\nDistribution of records per student-timestamp group:")
print(timestamp_stats.value_counts().sort_index().head(10))

Student-timestamp groups: 1915037
Groups with multiple records: 1727610
Percentage of groups with ties: 90.21 %
Records belonging to tied timestamp groups: 16029884
Percentage of all records in tied groups: 98.84 %

Distribution of records per student-timestamp group:
n_records
1     187427
2     200995
3     145941
4     152581
5     165926
6     159242
7     101986
8      97436
9      77227
10     83485
Name: count, dtype: int64


## Check original row order for chronological ordering

In [ ]:
# Preserve each record's original position in the source CSV
junyi_df["_original_row"] = np.arange(len(junyi_df))

# Look at records for one student that share a timestamp
example_student = model_df["uuid"].iloc[0]

student_rows = junyi_df[
    junyi_df["uuid"] == example_student
].copy()

student_rows["_timestamp_parsed"] = pd.to_datetime(
    student_rows["timestamp_TW"], utc=True, errors="coerce"
)

student_rows = student_rows.sort_values(
    ["_timestamp_parsed", "_original_row"],
    kind="mergesort"
)

display(
    student_rows[
        [
            "_original_row", "timestamp_TW", "ucid", "upid",
            "problem_number", "exercise_problem_repeat_session",
            "total_attempt_cnt", "is_correct"
        ]
    ].head(30)
)

print(
    "\nRecords for this student:",
    len(student_rows)
)
print(
    "Distinct timestamps:",
    student_rows["_timestamp_parsed"].nunique()
)

,_original_row,timestamp_TW,ucid,upid,problem_number,exercise_problem_repeat_session,total_attempt_cnt,is_correct
5851364,5851364,2018-12-22 08:30:00 UTC,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,wgsncTN/ZG/jV1BVhOb0ExV10Y7nbx4HDXgpqfV98Ls=,2,1,1,True
10682047,10682047,2018-12-22 08:30:00 UTC,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,uIHmyJc0Ia07OXiS1k2wayrEA57AKPTypHaTYtKc4Po=,5,1,1,True
11828870,11828870,2018-12-22 08:30:00 UTC,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,SZt4MEfoLNcDjOc4qWBsd5Lw5M03DA0f22Baij04cbg=,1,1,1,True
13798435,13798435,2018-12-22 08:30:00 UTC,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,hrdzJpcOlqUZbvMB6A0P2fHVvn6US4bPm05s/PsjRi4=,3,1,1,True
15461504,15461504,2018-12-22 08:30:00 UTC,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,s2T1bZCtwOT2xgCGkHFRiqlSX8MHEOiW36tBJRXYj4E=,4,1,1,True



Records for this student: 5
Distinct timestamps: 1


## Rebuild interaction tables with attempt counts

In [ ]:
# Preserve original row positions to align metadata safely
junyi_indexed = junyi_df.copy()
junyi_indexed["_source_row"] = np.arange(len(junyi_indexed))

# Recreate the cleaned interaction table with source-row identifiers
model_source = junyi_indexed[
    ["_source_row", "uuid", "ucid", "timestamp_TW", "is_correct",
     "total_attempt_cnt", "upid"]
].copy()

model_source["timestamp_TW"] = pd.to_datetime(
    model_source["timestamp_TW"], utc=True, errors="coerce"
)
model_source = model_source.dropna(
    subset=["uuid", "ucid", "timestamp_TW", "is_correct"]
)

model_source["is_correct"] = model_source["is_correct"].astype("int8")

# Reattach the existing IRT labels
model_source = model_source.merge(
    irt_labels,
    on="uuid",
    how="inner",
    validate="many_to_one"
)

# Reapply the existing student split assignments
split_lookup = {
    **{student: "train" for student in train_ids},
    **{student: "validation" for student in val_ids},
    **{student: "test" for student in test_ids},
}
model_source["split"] = model_source["uuid"].map(split_lookup)

# Reuse the existing skill mapping
model_source["skill_idx"] = model_source["ucid"].map(skill2idx)

print("Rows reconstructed:", len(model_source))
print("Rows with missing skill encoding:", model_source["skill_idx"].isna().sum())
print("Rows with missing attempt count:", model_source["total_attempt_cnt"].isna().sum())
print("\nSplit counts:")
print(model_source["split"].value_counts(dropna=False))

assert model_source["skill_idx"].notna().all()
assert model_source["split"].notna().all()
assert len(model_source) == len(model_df)

print("\nAlignment checks passed.")

Rows reconstructed: 16217311
Rows with missing skill encoding: 0
Rows with missing attempt count: 0

Split counts:
split
train         11675589
test           3268616
validation     1273106
Name: count, dtype: int64

Alignment checks passed.


## Create dataframes

In [ ]:
# Sort by student and timestamp.
# For tied timestamps, use problem ID and attempt count as
# deterministic tie-breakers; this does not claim true chronology.

model_source = model_source.sort_values(
    ["uuid", "timestamp_TW", "upid", "total_attempt_cnt", "_source_row"],
    kind="mergesort"
).reset_index(drop=True)

train_df = model_source[model_source["split"] == "train"].copy()
val_df = model_source[model_source["split"] == "validation"].copy()
test_df = model_source[model_source["split"] == "test"].copy()

# Verify records and student separation
print("=== Final DKT datasets ===")
for name, frame in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(
        f"{name}: {len(frame):,} rows | "
        f"{frame['uuid'].nunique():,} students | "
        f"{frame['skill_idx'].nunique():,} skills"
    )

assert len(train_df) + len(val_df) + len(test_df) == len(model_source)
assert set(train_df["uuid"]).isdisjoint(val_df["uuid"])
assert set(train_df["uuid"]).isdisjoint(test_df["uuid"])
assert set(val_df["uuid"]).isdisjoint(test_df["uuid"])

print("\nSplit and row-count checks passed.")

=== Final DKT datasets ===
Train: 11,675,589 rows | 52,385 students | 1,326 skills
Validation: 1,273,106 rows | 5,821 students | 1,326 skills
Test: 3,268,616 rows | 14,552 students | 1,326 skills

Split and row-count checks passed.


## Inspect sequence lengths for chunking

In [ ]:
# Count interactions per student in each split
def inspect_sequence_lengths(frame, name):
    lengths = frame.groupby("uuid", sort=False).size()

    print(f"\n=== {name} sequence lengths ===")
    print("Students:", len(lengths))
    print("Total interactions:", int(lengths.sum()))
    print("Minimum:", int(lengths.min()))
    print("Median:", int(lengths.median()))
    print("Mean:", round(lengths.mean(), 1))
    print("90th percentile:", int(lengths.quantile(0.90)))
    print("95th percentile:", int(lengths.quantile(0.95)))
    print("99th percentile:", int(lengths.quantile(0.99)))
    print("Maximum:", int(lengths.max()))

    return lengths

train_lengths = inspect_sequence_lengths(train_df, "Train")
val_lengths = inspect_sequence_lengths(val_df, "Validation")
test_lengths = inspect_sequence_lengths(test_df, "Test")


=== Train sequence lengths ===
Students: 52385
Total interactions: 11675589
Minimum: 1
Median: 59
Mean: 222.9
90th percentile: 588
95th percentile: 1004
99th percentile: 2240
Maximum: 10949

=== Validation sequence lengths ===
Students: 5821
Total interactions: 1273106
Minimum: 1
Median: 57
Mean: 218.7
90th percentile: 556
95th percentile: 926
99th percentile: 2313
Maximum: 7089

=== Test sequence lengths ===
Students: 14552
Total interactions: 3268616
Minimum: 1
Median: 60
Mean: 224.6
90th percentile: 598
95th percentile: 1025
99th percentile: 2241
Maximum: 11813


## Configure DKT sequence parameters

In [ ]:
# DKT sequence configuration
SEQ_LEN = 100
BATCH_SIZE = 128

NUM_SKILLS = len(skill2idx)

# DKT interaction encoding:
# input interaction = (skill_idx, response)
# We will encode skill-response pairs as skill_idx * 2 + response.
NUM_INTERACTION_TYPES = NUM_SKILLS * 2

print("Sequence length:", SEQ_LEN)
print("Batch size:", BATCH_SIZE)
print("Number of skills:", NUM_SKILLS)
print("Number of skill-response interaction types:", NUM_INTERACTION_TYPES)

Sequence length: 100
Batch size: 128
Number of skills: 1326
Number of skill-response interaction types: 2652


## Build chunks

In [ ]:
def build_dkt_chunks(frame, seq_len=100):
    chunks = []

    for student_id, group in frame.groupby("uuid", sort=False):
        group = group.sort_values(
            ["timestamp_TW", "upid", "total_attempt_cnt", "_source_row"],
            kind="mergesort"
        )

        skills = group["skill_idx"].to_numpy(dtype=np.int32)
        responses = group["is_correct"].to_numpy(dtype=np.int8)

        n = len(group)

        # Each chunk covers a disjoint range of target interactions.
        # For later chunks, prepend the previous interaction as context.
        for target_start in range(0, n, seq_len):
            target_end = min(target_start + seq_len, n)

            context_start = max(0, target_start - 1)

            chunk_skills = skills[context_start:target_end]
            chunk_responses = responses[context_start:target_end]

            # Inputs are all interactions except the final one.
            # Targets are the next interaction's skill and response.
            if len(chunk_skills) < 2:
                continue

            x = (
                chunk_skills[:-1] * 2
                + chunk_responses[:-1]
            ).astype(np.int32)

            target_skills = chunk_skills[1:].astype(np.int32)
            target_responses = chunk_responses[1:].astype(np.int8)

            # Mark which targets belong to this chunk's assigned range.
            target_positions = np.arange(
                context_start + 1,
                target_end
            )
            target_mask = (
                (target_positions >= target_start)
                & (target_positions < target_end)
            )

            chunks.append({
                "uuid": student_id,
                "x": x,
                "target_skills": target_skills,
                "target_responses": target_responses,
                "target_mask": target_mask,
            })

    return chunks


train_chunks = build_dkt_chunks(train_df, SEQ_LEN)
val_chunks = build_dkt_chunks(val_df, SEQ_LEN)
test_chunks = build_dkt_chunks(test_df, SEQ_LEN)

print("Train chunks:", len(train_chunks))
print("Validation chunks:", len(val_chunks))
print("Test chunks:", len(test_chunks))

print("\nExample chunk shapes:")
for name, chunks in [
    ("Train", train_chunks),
    ("Validation", val_chunks),
    ("Test", test_chunks),
]:
    example_chunk = chunks[0]
    print(
        f"{name}: inputs={len(example_chunk['x'])}, "
        f"targets={len(example_chunk['target_responses'])}, "
        f"masked targets={example_chunk['target_mask'].sum()}"
    )

Train chunks: 148312
Validation chunks: 16286
Test chunks: 41476

Example chunk shapes:
Train: inputs=4, targets=4, masked targets=4
Validation: inputs=94, targets=94, masked targets=94
Test: inputs=21, targets=21, masked targets=21


## Verify target coverage

In [ ]:
def verify_target_coverage(frame, chunks, name):
    expected_targets = len(frame) - frame["uuid"].nunique()
    actual_targets = sum(
        int(chunk["target_mask"].sum())
        for chunk in chunks
    )

    print(f"=== {name} ===")
    print("Expected next-step targets:", f"{expected_targets:,}")
    print("Actual targets in chunks:", f"{actual_targets:,}")
    print("Difference:", f"{actual_targets - expected_targets:,}")

    assert actual_targets == expected_targets, (
        f"{name}: target coverage mismatch"
    )

    assert all(
        len(chunk["x"]) == len(chunk["target_responses"])
        == len(chunk["target_skills"])
        == len(chunk["target_mask"])
        for chunk in chunks
    )

    print("Target coverage and shape checks passed.\n")


verify_target_coverage(train_df, train_chunks, "Train")
verify_target_coverage(val_df, val_chunks, "Validation")
verify_target_coverage(test_df, test_chunks, "Test")

=== Train ===
Expected next-step targets: 11,623,204
Actual targets in chunks: 11,623,204
Difference: 0
Target coverage and shape checks passed.

=== Validation ===
Expected next-step targets: 1,267,285
Actual targets in chunks: 1,267,285
Difference: 0
Target coverage and shape checks passed.

=== Test ===
Expected next-step targets: 3,254,064
Actual targets in chunks: 3,254,064
Difference: 0
Target coverage and shape checks passed.



## Define the DKT model

In [ ]:
class DKT(nn.Module):
    def __init__(self, num_interaction_types, num_skills,
                 embedding_dim=64, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(
            num_interaction_types, embedding_dim
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.output = nn.Linear(hidden_dim, num_skills)

    def forward(self, x):
        # x shape: (batch_size, sequence_length)
        embedded = self.embedding(x)
        hidden, _ = self.lstm(embedded)
        logits = self.output(hidden)

        # logits shape: (batch_size, sequence_length, num_skills)
        return logits


model = DKT(
    num_interaction_types=NUM_INTERACTION_TYPES,
    num_skills=NUM_SKILLS,
    embedding_dim=64,
    hidden_dim=128
).to(device)

print(model)
print("\nDevice:", next(model.parameters()).device)
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

DKT(
  (embedding): Embedding(2652, 64)
  (lstm): LSTM(64, 128, batch_first=True)
  (output): Linear(in_features=128, out_features=1326, bias=True)
)

Device: cuda:0
Trainable parameters: 440,110


## Build the padded batch loader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class DKTChunkDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        return (
            torch.tensor(chunk["x"], dtype=torch.long),
            torch.tensor(chunk["target_skills"], dtype=torch.long),
            torch.tensor(chunk["target_responses"], dtype=torch.float32),
            torch.tensor(chunk["target_mask"], dtype=torch.bool),
        )


def collate_dkt(batch):
    xs, target_skills, target_responses, target_masks = zip(*batch)

    x_pad = pad_sequence(xs, batch_first=True, padding_value=0)
    skill_pad = pad_sequence(
        target_skills, batch_first=True, padding_value=0
    )
    response_pad = pad_sequence(
        target_responses, batch_first=True, padding_value=0
    )
    mask_pad = pad_sequence(
        target_masks, batch_first=True, padding_value=False
    )

    # Only real target positions are included in the loss.
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)

    return x_pad, skill_pad, response_pad, mask_pad, lengths


train_loader = DataLoader(
    DKTChunkDataset(train_chunks),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_dkt,
    num_workers=0,
)

val_loader = DataLoader(
    DKTChunkDataset(val_chunks),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_dkt,
    num_workers=0,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

x_batch, skill_batch, response_batch, mask_batch, lengths = next(iter(train_loader))

print("\nBatch shapes:")
print("Input:", tuple(x_batch.shape))
print("Target skills:", tuple(skill_batch.shape))
print("Target responses:", tuple(response_batch.shape))
print("Target mask:", tuple(mask_batch.shape))
print("Valid targets in batch:", int(mask_batch.sum()))

Train batches: 1159
Validation batches: 128

Batch shapes:
Input: (128, 100)
Target skills: (128, 100)
Target responses: (128, 100)
Target mask: (128, 100)
Valid targets in batch: 10414


## Rebuild chunks with sufficient historical context

In [ ]:
def build_dkt_chunks(frame, seq_len=100):
    chunks = []

    for student_id, group in frame.groupby("uuid", sort=False):
        group = group.sort_values(
            ["timestamp_TW", "upid", "total_attempt_cnt", "_source_row"],
            kind="mergesort"
        )

        skills = group["skill_idx"].to_numpy(dtype=np.int32)
        responses = group["is_correct"].to_numpy(dtype=np.int8)
        n = len(group)

        # Assign each next-step target to exactly one chunk.
        for target_start in range(0, n, seq_len):
            target_end = min(target_start + seq_len, n)

            # Retain up to seq_len previous interactions as context.
            context_start = max(0, target_start - seq_len)

            chunk_skills = skills[context_start:target_end]
            chunk_responses = responses[context_start:target_end]

            if len(chunk_skills) < 2:
                continue

            x = (
                chunk_skills[:-1] * 2 + chunk_responses[:-1]
            ).astype(np.int32)

            target_skills = chunk_skills[1:].astype(np.int32)
            target_responses = chunk_responses[1:].astype(np.int8)

            target_positions = np.arange(
                context_start + 1, target_end
            )

            # Only score targets assigned to this chunk.
            target_mask = (
                (target_positions >= target_start)
                & (target_positions < target_end)
            )

            chunks.append({
                "uuid": student_id,
                "x": x,
                "target_skills": target_skills,
                "target_responses": target_responses,
                "target_mask": target_mask,
            })

    return chunks


# Rebuild all chunks with historical context
train_chunks = build_dkt_chunks(train_df, SEQ_LEN)
val_chunks = build_dkt_chunks(val_df, SEQ_LEN)
test_chunks = build_dkt_chunks(test_df, SEQ_LEN)

# Recreate loaders because chunk lengths have changed
train_loader = DataLoader(
    DKTChunkDataset(train_chunks),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_dkt,
    num_workers=0,
)

val_loader = DataLoader(
    DKTChunkDataset(val_chunks),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_dkt,
    num_workers=0,
)

test_loader = DataLoader(
    DKTChunkDataset(test_chunks),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_dkt,
    num_workers=0,
)

print("Train chunks:", len(train_chunks))
print("Validation chunks:", len(val_chunks))
print("Test chunks:", len(test_chunks))

# Verify that each eligible next-step target is scored exactly once
verify_target_coverage(train_df, train_chunks, "Train")
verify_target_coverage(val_df, val_chunks, "Validation")
verify_target_coverage(test_df, test_chunks, "Test")

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train chunks: 148312
Validation chunks: 16286
Test chunks: 41476
=== Train ===
Expected next-step targets: 11,623,204
Actual targets in chunks: 11,623,204
Difference: 0
Target coverage and shape checks passed.

=== Validation ===
Expected next-step targets: 1,267,285
Actual targets in chunks: 1,267,285
Difference: 0
Target coverage and shape checks passed.

=== Test ===
Expected next-step targets: 3,254,064
Actual targets in chunks: 3,254,064
Difference: 0
Target coverage and shape checks passed.

Train batches: 1159
Validation batches: 128


## Define the training loss

In [ ]:
def dkt_loss(logits, target_skills, target_responses, target_mask):
    # logits: (batch, sequence, num_skills)
    # Select the prediction for each target's actual skill.
    selected_logits = logits.gather(
        dim=2,
        index=target_skills.unsqueeze(-1)
    ).squeeze(-1)

    valid_logits = selected_logits[target_mask]
    valid_targets = target_responses[target_mask]

    loss = F.binary_cross_entropy_with_logits(
        valid_logits,
        valid_targets
    )

    return loss


# Test one batch
model.train()
batch = next(iter(train_loader))
x_batch, skill_batch, response_batch, mask_batch, lengths = batch

x_batch = x_batch.to(device)
skill_batch = skill_batch.to(device)
response_batch = response_batch.to(device)
mask_batch = mask_batch.to(device)

with torch.no_grad():
    logits = model(x_batch)
    loss = dkt_loss(logits, skill_batch, response_batch, mask_batch)

print("Logits shape:", tuple(logits.shape))
print("Valid targets:", int(mask_batch.sum()))
print("Test batch loss:", round(loss.item(), 5))
print("Loss finite:", bool(torch.isfinite(loss)))

Logits shape: (128, 100, 1326)
Valid targets: 10146
Test batch loss: 0.51959
Loss finite: True


## Train the model

In [ ]:
import copy
from sklearn.metrics import roc_auc_score, brier_score_loss

MODEL_PATH = os.path.join(RESULTS_DIR, "junyi_dkt_best.pt")
EPOCHS = 10
PATIENCE = 3
LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def evaluate_dkt(model, loader):
    model.eval()
    all_probs = []
    all_targets = []
    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for x, skills, responses, mask, lengths in loader:
            x = x.to(device)
            skills = skills.to(device)
            responses = responses.to(device)
            mask = mask.to(device)

            logits = model(x)
            selected_logits = logits.gather(
                2, skills.unsqueeze(-1)
            ).squeeze(-1)

            valid_logits = selected_logits[mask]
            valid_targets = responses[mask]

            loss = F.binary_cross_entropy_with_logits(
                valid_logits, valid_targets, reduction="sum"
            )

            probs = torch.sigmoid(valid_logits.float())

            all_probs.append(probs.cpu().numpy())
            all_targets.append(valid_targets.cpu().numpy())
            total_loss += loss.item()
            total_count += valid_targets.numel()

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probs)

    return {
        "loss": total_loss / total_count,
        "auc": roc_auc_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "n_targets": total_count,
    }


best_val_auc = -float("inf")
best_epoch = None
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_targets = 0

    for x, skills, responses, mask, lengths in train_loader:
        x = x.to(device)
        skills = skills.to(device)
        responses = responses.to(device)
        mask = mask.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = dkt_loss(logits, skills, responses, mask)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        n_valid = int(mask.sum().item())
        running_loss += loss.item() * n_valid
        running_targets += n_valid

    train_loss = running_loss / running_targets
    val_metrics = evaluate_dkt(model, val_loader)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_auc": val_metrics["auc"],
        "val_brier": val_metrics["brier"],
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train loss={train_loss:.5f} | "
        f"val loss={val_metrics['loss']:.5f} | "
        f"val AUC={val_metrics['auc']:.5f} | "
        f"val Brier={val_metrics['brier']:.5f}"
    )

    if val_metrics["auc"] > best_val_auc:
        best_val_auc = val_metrics["auc"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        torch.save({
            "epoch": best_epoch,
            "model_state_dict": best_state,
            "val_auc": best_val_auc,
            "num_skills": NUM_SKILLS,
            "num_interaction_types": NUM_INTERACTION_TYPES,
            "embedding_dim": 64,
            "hidden_dim": 128,
            "seq_len": SEQ_LEN,
            "seed": SEED,
        }, MODEL_PATH)

        print("  Saved new best checkpoint.")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping after epoch {epoch}.")
        break

model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_path = os.path.join(RESULTS_DIR, "junyi_dkt_training_history.csv")
history_df.to_csv(history_path, index=False)

print("\nBest epoch:", best_epoch)
print("Best validation AUC:", round(best_val_auc, 6))
print("Checkpoint:", MODEL_PATH)
print("Training history:", history_path)

In [ ]:
# Evaluate the model before any training updates
initial_train = evaluate_dkt(model, train_loader)
initial_val = evaluate_dkt(model, val_loader)

print("Initial model — before training")
print(
    f"Train: AUC={initial_train['auc']:.5f}, "
    f"Loss={initial_train['loss']:.5f}, "
    f"Brier={initial_train['brier']:.5f}"
)
print(
    f"Val:   AUC={initial_val['auc']:.5f}, "
    f"Loss={initial_val['loss']:.5f}, "
    f"Brier={initial_val['brier']:.5f}"
)

Initial model — before training
Train: AUC=0.81410, Loss=0.46406, Brier=0.15177
Val:   AUC=0.72605, Loss=0.56007, Brier=0.18554


## Clear the optimizer

In [ ]:
# Create a fresh DKT model and optimizer
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

fresh_model = DKT(
    num_interaction_types=NUM_INTERACTION_TYPES,
    num_skills=NUM_SKILLS,
    embedding_dim=64,
    hidden_dim=128
).to(device)

fresh_optimizer = torch.optim.Adam(
    fresh_model.parameters(),
    lr=1e-3
)

print("Fresh model created.")
print("Optimizer state entries:", len(fresh_optimizer.state))

Fresh model created.
Optimizer state entries: 0


## Train after clearing the optimizer

In [ ]:
MODEL_PATH = os.path.join(RESULTS_DIR, "junyi_dkt_best.pt")
HISTORY_PATH = os.path.join(RESULTS_DIR, "junyi_dkt_training_history.csv")

MAX_EPOCHS = 100
best_val_auc = -float("inf")
best_epoch = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    fresh_model.train()
    total_loss = 0.0
    total_targets = 0

    for x, skills, responses, mask, lengths in train_loader:
        x = x.to(device)
        skills = skills.to(device)
        responses = responses.to(device)
        mask = mask.to(device)

        fresh_optimizer.zero_grad()
        logits = fresh_model(x)
        loss = dkt_loss(logits, skills, responses, mask)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            fresh_model.parameters(), max_norm=5.0
        )
        fresh_optimizer.step()

        n = responses[mask].numel()
        total_loss += loss.item() * n
        total_targets += n

    train_loss = total_loss / total_targets
    val_metrics = evaluate_dkt(fresh_model, val_loader)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_auc": val_metrics["auc"],
        "val_brier": val_metrics["brier"],
    })

    improved = val_metrics["auc"] > best_val_auc

    if improved:
        best_val_auc = val_metrics["auc"]
        best_epoch = epoch

        torch.save({
            "epoch": epoch,
            "model_state_dict": fresh_model.state_dict(),
            "optimizer_state_dict": fresh_optimizer.state_dict(),
            "val_auc": best_val_auc,
            "val_loss": val_metrics["loss"],
            "val_brier": val_metrics["brier"],
        }, MODEL_PATH)

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch:03d}/100 | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_metrics['loss']:.5f} | "
        f"Val AUC: {val_metrics['auc']:.5f} | "
        f"Val Brier: {val_metrics['brier']:.5f} | "
        f"{'BEST' if improved else ''}"
    )

print("\nTraining complete.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation AUC: {best_val_auc:.5f}")
print(f"Best checkpoint: {MODEL_PATH}")
print(f"Training history: {HISTORY_PATH}")

Epoch 001/100 | Train Loss: 0.53731 | Val Loss: 0.52320 | Val AUC: 0.75198 | Val Brier: 0.17431 | BEST
Epoch 002/100 | Train Loss: 0.51599 | Val Loss: 0.51869 | Val AUC: 0.75746 | Val Brier: 0.17253 | BEST
Epoch 003/100 | Train Loss: 0.51232 | Val Loss: 0.51708 | Val AUC: 0.75939 | Val Brier: 0.17194 | BEST
Epoch 004/100 | Train Loss: 0.51023 | Val Loss: 0.51663 | Val AUC: 0.75997 | Val Brier: 0.17177 | BEST
Epoch 005/100 | Train Loss: 0.50863 | Val Loss: 0.51638 | Val AUC: 0.76035 | Val Brier: 0.17167 | BEST
Epoch 006/100 | Train Loss: 0.50722 | Val Loss: 0.51652 | Val AUC: 0.76013 | Val Brier: 0.17175 | 
Epoch 007/100 | Train Loss: 0.50591 | Val Loss: 0.51654 | Val AUC: 0.76025 | Val Brier: 0.17177 | 
Epoch 008/100 | Train Loss: 0.50470 | Val Loss: 0.51693 | Val AUC: 0.75972 | Val Brier: 0.17189 | 
Epoch 009/100 | Train Loss: 0.50353 | Val Loss: 0.51752 | Val AUC: 0.75929 | Val Brier: 0.17214 | 
Epoch 010/100 | Train Loss: 0.50245 | Val Loss: 0.51791 | Val AUC: 0.75880 | Val Brier: 0

## Eval on epoch 5

In [ ]:
# Junyi DKT: held-out test evaluation using the saved best checkpoint

import torch
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss,
    accuracy_score
)

checkpoint_path = (
    "/content/drive/MyDrive/education-ml-research/Junyi/"
    "reliability_experiment_irt/junyi_dkt_best.pt"
)

# Load checkpoint safely for your own trusted training artifact.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Recover model weights from common checkpoint formats.
if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict)
model.to(device)
model.eval()

all_probs = []
all_targets = []

with torch.no_grad():
    for q_in, r_in, q_target, mask, student_ids in test_loader:
        q_in = q_in.to(device)
        r_in = r_in.to(device)
        q_target = q_target.to(device)
        mask = mask.to(device)

        # Use the input encoding already used by the dataset.
        # Do NOT combine q_in and r_in again.
        outputs = model(q_in)

        # Convert encoded interaction IDs to skill IDs.
        target_skill_ids = torch.div(q_in, 2, rounding_mode="floor")

        # Select the logit corresponding to each target skill.
        logits = outputs.gather(
            dim=2,
            index=target_skill_ids.unsqueeze(-1)
        ).squeeze(-1)

        probs = torch.sigmoid(logits)

        valid_probs = probs[mask].detach().cpu().numpy()
        valid_targets = q_target[mask].detach().cpu().numpy()

        all_probs.extend(valid_probs.tolist())
        all_targets.extend(valid_targets.tolist())

all_probs = np.asarray(all_probs)
all_targets = np.asarray(all_targets).astype(int)

valid = np.isfinite(all_probs) & np.isfinite(all_targets)
all_probs = np.clip(all_probs[valid], 1e-7, 1 - 1e-7)
all_targets = all_targets[valid]

print("Junyi DKT — Held-out Test Results")
print("--------------------------------")
print(f"Checkpoint: {checkpoint_path}")
print(f"Valid predictions: {len(all_targets):,}")
print(f"Test AUC:      {roc_auc_score(all_targets, all_probs):.6f}")
print(f"Test Brier:    {brier_score_loss(all_targets, all_probs):.6f}")
print(f"Test Log Loss: {log_loss(all_targets, all_probs):.6f}")
print(f"Test Accuracy: {accuracy_score(all_targets, all_probs >= 0.5):.6f}")

Junyi DKT — Held-out Test Results
--------------------------------
Checkpoint: /content/drive/MyDrive/education-ml-research/Junyi/reliability_experiment_irt/junyi_dkt_best.pt
Valid predictions: 3,254,064
Test AUC:      0.716372
Test Brier:    0.183155
Test Log Loss: 0.549544
Test Accuracy: 0.731903


## Eval on IRT groups

In [ ]:
# Junyi DKT — held-out test results by IRT ability group
# Uses UUIDs directly from DKTChunkDataset.chunks.

import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss,
    accuracy_score
)

base = "/content/drive/MyDrive/education-ml-research/Junyi"
checkpoint_path = os.path.join(
    base, "reliability_experiment_irt", "junyi_dkt_best.pt"
)
irt_path = os.path.join(base, "student_irt_ability_groups.csv")

# --------------------------------------------------
# Load the existing IRT group assignments
# --------------------------------------------------
irt_groups = pd.read_csv(irt_path, dtype={"uuid": str})

required = {"uuid", "theta_all", "theta_quartile", "theta_sd_group"}
if not required.issubset(irt_groups.columns):
    raise ValueError(f"Missing IRT columns: {required - set(irt_groups.columns)}")

if irt_groups["uuid"].duplicated().any():
    raise ValueError("IRT group file contains duplicate UUIDs.")

irt_groups["uuid"] = irt_groups["uuid"].str.strip()

# --------------------------------------------------
# Load the existing best checkpoint
# --------------------------------------------------
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict)
model.to(device)
model.eval()

# --------------------------------------------------
# Evaluate the existing held-out test chunks
# --------------------------------------------------
test_chunks = test_loader.dataset.chunks
rows = []

with torch.no_grad():
    for start in range(0, len(test_chunks), 128):
        batch = test_chunks[start:start + 128]

        # These are the actual inputs and targets stored in the dataset.
        x = torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(c["x"], dtype=torch.long) for c in batch],
            batch_first=True,
            padding_value=0
        ).to(device)

        target_skills = torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(c["target_skills"], dtype=torch.long) for c in batch],
            batch_first=True,
            padding_value=0
        ).to(device)

        targets = torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(c["target_responses"], dtype=torch.float32) for c in batch],
            batch_first=True,
            padding_value=0
        ).to(device)

        masks = torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(c["target_mask"], dtype=torch.bool) for c in batch],
            batch_first=True,
            padding_value=False
        ).to(device)

        # DKT outputs one logit per skill at every sequence position.
        outputs = model(x)

        if outputs.ndim != 3:
            raise ValueError(f"Expected [batch, seq_len, skills] outputs; got {outputs.shape}")

        if outputs.shape[:2] != target_skills.shape:
            raise ValueError(
                f"Sequence mismatch: outputs {outputs.shape}, "
                f"target skills {target_skills.shape}"
            )

        if target_skills.numel() and (
            target_skills.min().item() < 0
            or target_skills.max().item() >= outputs.shape[-1]
        ):
            raise ValueError("Target skill index is outside the model output range.")

        # Select the logit corresponding to each recorded target skill.
        logits = outputs.gather(
            2, target_skills.unsqueeze(-1)
        ).squeeze(-1)

        probs = torch.sigmoid(logits).cpu().numpy()
        y = targets.cpu().numpy()
        valid = masks.cpu().numpy()

        for i, chunk in enumerate(batch):
            keep = valid[i] & np.isfinite(probs[i]) & np.isfinite(y[i])

            if not keep.any():
                continue

            rows.append(pd.DataFrame({
                "uuid": str(chunk["uuid"]).strip(),
                "y_true": y[i, keep].astype(int),
                "y_prob": probs[i, keep]
            }))

if not rows:
    raise ValueError("No valid test predictions were generated.")

pred_df = pd.concat(rows, ignore_index=True)

# --------------------------------------------------
# Match the predictions to the exact IRT groups
# --------------------------------------------------
pred_df = pred_df.merge(
    irt_groups[["uuid", "theta_all", "theta_quartile", "theta_sd_group"]],
    on="uuid",
    how="left",
    validate="many_to_one",
    indicator=True
)

unmatched = pred_df.loc[pred_df["_merge"] != "both", "uuid"].nunique()
print("Unique test students:", pred_df["uuid"].nunique())
print("Unique students matched to IRT:", pred_df.loc[pred_df["_merge"] == "both", "uuid"].nunique())
print("Unique unmatched test students:", unmatched)

pred_df = pred_df.loc[pred_df["_merge"] == "both"].drop(columns="_merge")

if pred_df.empty:
    raise ValueError("No test UUIDs matched the IRT group file.")

# --------------------------------------------------
# Compute metrics
# --------------------------------------------------
def metrics(g):
    y = g["y_true"].to_numpy()
    p = np.clip(g["y_prob"].to_numpy(), 1e-7, 1 - 1e-7)

    return pd.Series({
        "Students": g["uuid"].nunique(),
        "Predictions": len(g),
        "Correct Rate": y.mean(),
        "AUC": roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
        "Brier": brier_score_loss(y, p),
        "Log Loss": log_loss(y, p, labels=[0, 1]),
        "Accuracy": accuracy_score(y, p >= 0.5)
    })

quartile_order = ["Q1", "Q2", "Q3", "Q4"]
sd_order = [
    "< -1.5 SD",
    "-1.5 to -1.0 SD",
    "-1.0 to -0.5 SD",
    "-0.5 to 0 SD",
    "0 to +0.5 SD",
    "+0.5 to +1.0 SD",
    "+1.0 to +1.5 SD",
    "> +1.5 SD"
]

quartile_results = (
    pred_df.groupby("theta_quartile", observed=True)
    .apply(metrics, include_groups=False)
    .reindex(quartile_order)
)

sd_results = (
    pred_df.groupby("theta_sd_group", observed=True)
    .apply(metrics, include_groups=False)
    .reindex(sd_order)
)

print("\n=== Junyi DKT: Held-out Results by IRT Quartile ===")
display(quartile_results.round(6))

print("\n=== Junyi DKT: Held-out Results by IRT SD Group ===")
display(sd_results.round(6))

Unique test students: 14210
Unique students matched to IRT: 14210
Unique unmatched test students: 0

=== Junyi DKT: Held-out Results by IRT Quartile ===


,Students,Predictions,Correct Rate,AUC,Brier,Log Loss,Accuracy
theta_quartile,,,,,,,
Q1,3432.0,718127.0,0.548392,0.744644,0.204276,0.594032,0.679288
Q2,3623.0,1042418.0,0.680124,0.739329,0.183806,0.545695,0.722836
Q3,3545.0,871242.0,0.761625,0.734519,0.158169,0.484099,0.773551
Q4,3610.0,622277.0,0.835832,0.720914,0.125920,0.405767,0.835343



=== Junyi DKT: Held-out Results by IRT SD Group ===


,Students,Predictions,Correct Rate,AUC,Brier,Log Loss,Accuracy
theta_sd_group,,,,,,,
< -1.5 SD,1027.0,125528.0,0.437775,0.734555,0.209738,0.607584,0.668353
-1.5 to -1.0 SD,1317.0,312538.0,0.550100,0.735981,0.207043,0.600232,0.672497
-1.0 to -0.5 SD,2325.0,612582.0,0.625252,0.744220,0.194374,0.570782,0.701101
-0.5 to 0 SD,3032.0,881540.0,0.701900,0.738032,0.178145,0.532097,0.733947
0 to +0.5 SD,2785.0,675956.0,0.768333,0.732518,0.155867,0.478585,0.778286
+0.5 to +1.0 SD,1910.0,413375.0,0.813644,0.723465,0.136914,0.432931,0.815354
+1.0 to +1.5 SD,1062.0,174544.0,0.854209,0.705179,0.117617,0.385918,0.851820
> +1.5 SD,752.0,58001.0,0.914139,0.694094,0.083200,0.298187,0.908588


## Junyi DKT: Overall Held-out Test Results

| Metric | Value |
|---|---:|
| Valid Predictions | 3,254,064 |
| Test AUC | 0.716372 |
| Test Brier Score | 0.183155 |
| Test Log Loss | 0.549544 |
| Test Accuracy | 0.731903 |

## Junyi DKT: Held-out Results by IRT Quartile

| IRT Quartile | Students | Predictions | Correct Rate | AUC | Brier | Log Loss | Accuracy |
|---|---:|---:|---:|---:|---:|---:|---:|
| Q1 | 3,432 | 718,127 | 0.548392 | 0.744644 | 0.204276 | 0.594032 | 0.679288 |
| Q2 | 3,623 | 1,042,418 | 0.680124 | 0.739329 | 0.183806 | 0.545695 | 0.722836 |
| Q3 | 3,545 | 871,242 | 0.761625 | 0.734519 | 0.158169 | 0.484099 | 0.773551 |
| Q4 | 3,610 | 622,277 | 0.835832 | 0.720914 | 0.125920 | 0.405767 | 0.835343 |

## Junyi DKT: Held-out Results by IRT Standard-Deviation Group

| IRT SD Group | Students | Predictions | Correct Rate | AUC | Brier | Log Loss | Accuracy |
|---|---:|---:|---:|---:|---:|---:|---:|
| < -1.5 SD | 1,027 | 125,528 | 0.437775 | 0.734555 | 0.209738 | 0.607584 | 0.668353 |
| -1.5 to -1.0 SD | 1,317 | 312,538 | 0.550100 | 0.735981 | 0.207043 | 0.600232 | 0.672497 |
| -1.0 to -0.5 SD | 2,325 | 612,582 | 0.625252 | 0.744220 | 0.194374 | 0.570782 | 0.701101 |
| -0.5 to 0 SD | 3,032 | 881,540 | 0.701900 | 0.738032 | 0.178145 | 0.532097 | 0.733947 |
| 0 to +0.5 SD | 2,785 | 675,956 | 0.768333 | 0.732518 | 0.155867 | 0.478585 | 0.778286 |
| +0.5 to +1.0 SD | 1,910 | 413,375 | 0.813644 | 0.723465 | 0.136914 | 0.432931 | 0.815354 |
| +1.0 to +1.5 SD | 1,062 | 174,544 | 0.854209 | 0.705179 | 0.117617 | 0.385918 | 0.851820 |
| > +1.5 SD | 752 | 58,001 | 0.914139 | 0.694094 | 0.083200 | 0.298187 | 0.908588 |